# Graph Link Prediction Pipeline

This notebook demonstrates the complete two-stage link prediction pipeline:

**Stage 1 (Task 1): SAME_DENOMINATOR** - Partition nodes by rational value μ = n/d

**Stage 2 (Task 2): NEXT_INTEGER** - Order nodes by consecutive integers within partitions

We implement and compare all approaches:
- Task 1: Classifier, Pairwise, Contrastive
- Task 2: Regressor, Pairwise


## 1. Setup and Imports


In [1]:
import sys
sys.path.insert(0, '..')

import torch
import numpy as np

from graph_link_prediction import (
    load_graph_from_neo4j,
    TwoStageLinkPredictionPipeline,
    Task1Trainer,
    Task2Trainer,
    LinkPredictor,
)

print("Imports successful!")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


Imports successful!
PyTorch version: 2.9.1+cpu
CUDA available: False


## 2. Load Data from Neo4j


In [2]:
# Configure Neo4j connection


NEO4J_URI="bolt://localhost:7687"
NEO4J_USER="neo4j"
NEO4J_PASSWORD="ChiefQuippy"
NEO4J_DATABASE="d4seed1"


# Load data
print("Loading data from Neo4j...")
data_task1, data_task2, loader = load_graph_from_neo4j(
    uri=NEO4J_URI,
    user=NEO4J_USER,
    password=NEO4J_PASSWORD,
    database=NEO4J_DATABASE,
)

print("\nData loaded successfully!")


Loading data from Neo4j...
Loading data from Neo4j...


Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: unknown property key. The property `nBinary` does not exist. Verify that the spelling is correct.', position=<SummaryInputPosition line=9, column=19, offset=184>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_status_parameters': {'propkey': 'nBinary'}, '_severity': 'WARNING', '_position': {'offset': 184, 'line': 9, 'column': 19}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\nMATCH (d:Dnode)\nWHERE d.determined = 1\nRETURN elementId(d) as node_id,\n       d.muList as muList,\n       d.n as n,\n       d.d as d,\n       d.totalZero as totalZero,\n       coalesce(d.nBinary, '') as nBinary\nORDER BY node_id\n"


  Found 2143 determined nodes
  Computing integer values from muList...
  Computing partition IDs...
  Loading wNum values...
  Loading zMap edges...
  Generating ground truth edges...
  Generated 2 SAME_DENOMINATOR edges
  Generated 2 NEXT_INTEGER edges
  Building Task 1 features...
  Building Task 2 features...
Data loading complete!

Dataset Statistics:
  Nodes: 2143
  Structural edges (zMap): 1673
  SAME_DENOMINATOR edges: 36258
  NEXT_INTEGER edges: 0

Partition Statistics:
  Number of partitions: 496
  Mean partition size: 4.32
  Max partition size: 37

Integer Value Statistics:
  Range: [0, 562949953421824]
  Unique integers: 519


Data loaded successfully!


## 3. Run Complete Pipeline


In [3]:
# Create pipeline
pipeline = TwoStageLinkPredictionPipeline(data_task1, data_task2)

# Run with default approaches
results = pipeline.run(
    task1_approach='classifier',
    task2_approach='regressor',
    verbose=True
)

# Print report
from graph_link_prediction.config import TASK1_TARGETS, TASK2_TARGETS
report = pipeline.generate_report(TASK1_TARGETS, TASK2_TARGETS)
print(report)


Task 1 Trainer initialized:
  Nodes: 2143
  Partitions: 496
  Ground truth edges: 36258
  Device: cpu

Task 2 Trainer initialized:
  Nodes: 2143
  Partitions: 496
  Ground truth NEXT_INTEGER edges: 0
  Device: cpu


TWO-STAGE LINK PREDICTION PIPELINE
Stage 1 Approach: classifier
Stage 2 Approach: regressor

STAGE 1: Training SAME_DENOMINATOR predictor...
Training Task 1: Partition Classifier
Epoch 000 | Loss: 8.0187 | Acc: 0.0023
Epoch 010 | Loss: 5.6254 | Acc: 0.0163
Epoch 020 | Loss: 5.3694 | Acc: 0.0322
Epoch 030 | Loss: 5.1702 | Acc: 0.0401
Epoch 040 | Loss: 5.0486 | Acc: 0.0434
Epoch 050 | Loss: 4.7585 | Acc: 0.0527
Epoch 060 | Loss: 4.4459 | Acc: 0.0737
Epoch 070 | Loss: 4.4313 | Acc: 0.0751
Epoch 080 | Loss: 4.4051 | Acc: 0.0831
Epoch 090 | Loss: 4.3914 | Acc: 0.0831

Results:
  Partition Purity: 0.2783
  Adjusted Rand Index: 0.0706
  Link F1: 0.0843

STAGE 2: Training NEXT_INTEGER predictor...
Training Task 2: Integer Value Regressor
Epoch 000 | Loss: 17664428725589973757696409

## 4. Compare All Approaches


In [4]:
# Compare all approach combinations
comparison_results = pipeline.compare_approaches(
    task1_approaches=['classifier', 'pairwise', 'contrastive'],
    task2_approaches=['regressor', 'pairwise'],
    verbose=True
)

print("\nBest combination:", comparison_results['best_combination'])
print(f"Best F1: {comparison_results['best_result']['combined_f1']:.4f}")



COMPARING ALL APPROACH COMBINATIONS


Testing: classifier+regressor...
  Task 1 F1: 0.0948
  Task 2 F1: 0.0000
  Combined F1: 0.0474
  Bijection Valid: False

Testing: classifier+pairwise...
  Task 1 F1: 0.0532
  Task 2 F1: 0.0000
  Combined F1: 0.0266
  Bijection Valid: False

Testing: pairwise+regressor...
  Task 1 F1: 0.0000
  Task 2 F1: 0.0000
  Combined F1: 0.0000
  Bijection Valid: False

Testing: pairwise+pairwise...
  Task 1 F1: 0.0000
  Task 2 F1: 0.0000
  Combined F1: 0.0000
  Bijection Valid: True

Testing: contrastive+regressor...
  Task 1 F1: 0.1105
  Task 2 F1: 0.0000
  Combined F1: 0.0553
  Bijection Valid: False

Testing: contrastive+pairwise...
  Task 1 F1: 0.1112
  Task 2 F1: 0.0000
  Combined F1: 0.0556
  Bijection Valid: False

BEST COMBINATION: contrastive+pairwise
  Combined F1: 0.0556
  Bijection Valid: False


Best combination: contrastive+pairwise
Best F1: 0.0556
